<a href="https://colab.research.google.com/github/financieras/articulos/blob/main/logistic_regression_english_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Logistic Regression: Manual vs Scikit-learn**
## **Complete beginner's guide: from Gradient Descent to professional implementation in medical diagnosis**

---

## Table of Contents

1. Introduction: What is Logistic Regression?
2. Data Preparation
3. Model Fundamentals
4. Manual Implementation with Gradient Descent
5. Manual Prediction and Evaluation
6. Implementation with Scikit-learn
7. Conclusions

---

## **1. Introduction: What is Logistic Regression?**

**Logistic Regression** is one of the most fundamental and widely used algorithms in Machine Learning for **classification** problems. Despite its name, it is not used for regression but for classification: predicting categories or classes.

### Why is it called "Regression" if it's Classification?

The name may seem confusing at first. Logistic regression is called this way because internally it uses a linear regression function, but then transforms its output through the **sigmoid function** to obtain probabilities between 0 and 1. These probabilities are finally converted into discrete classes.

### Key Differences with Linear Regression

| Aspect | Linear Regression | Logistic Regression |
|---------|------------------|---------------------|
| **Objective** | Predict continuous values | Predict categories/classes |
| **Output** | Any real number | Probability between 0 and 1 |
| **Example** | Predict house price | Predict if a tumor is malignant or benign |
| **Function** | $y = w^T x + b$ | $P(y=1) = \sigma(w^T x + b)$ |

### Our Use Case: Breast Cancer Diagnosis

In this article we will work with the **Wisconsin Breast Cancer Dataset**, one of the most widely used datasets in medical Machine Learning. Our goal is to classify tumors as:

- **Class 0**: Malignant (cancerous)
- **Class 1**: Benign (non-cancerous)

The dataset contains 30 numerical features calculated from digitized images of fine needle aspirates of breast masses. These features describe properties of cell nuclei present in the image.

### What Will You Learn in This Article?

In this complete beginner's tutorial you will learn:

1. **The theoretical foundations** of logistic regression with accessible mathematics
2. **Implement the algorithm from scratch** using Gradient Descent
3. **Manually split data** into training and test sets
4. **Evaluate your model** by calculating Accuracy and understanding what it means
5. **Implement the professional solution** using Scikit-learn
6. **Compare both approaches** and understand when to use each one
7. **Extend the concept** to multiclass classification
8. **Understand the variants** of Gradient Descent (GD, SGD, Mini-Batch)

### Why is Logistic Regression Important?

Although there are more sophisticated algorithms like Neural Networks or Gradient Boosting, Logistic Regression remains extremely valuable because of:

- **Simplicity**: Easy to implement and understand
- **Interpretability**: The weights tell us which features are most important
- **Efficiency**: Very fast to train, even with large datasets
- **Solid baseline**: Excellent starting point before trying complex models
- **Competitive results**: In many linearly separable problems, it achieves accuracy similar to more complex models

**Let's get started!** In the next section we will load and explore our medical dataset.

---

## **2. Data Preparation**

In this section we will load the Wisconsin Breast Cancer dataset, explore its main characteristics, perform the manual split into training and test sets, and normalize the data to ensure stable training.

### 2.1 Loading the Dataset

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

# Load the dataset
data = load_breast_cancer()

# Create arrays of features and labels
X = data.data      # 569 samples x 30 features
y = data.target    # 569 labels (0: malignant, 1: benign)

# Create DataFrame for better visualization (optional)
df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y

print(f"Dataset shape: {X.shape}")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")

Dataset shape: (569, 30)
Number of samples: 569
Number of features: 30


### 2.2 Basic Exploration

In [2]:
# Class distribution
print("\nClass Distribution:")
print(f"Malignant (0): {np.sum(y == 0)} samples ({np.sum(y == 0)/len(y)*100:.1f}%)")
print(f"Benign (1): {np.sum(y == 1)} samples ({np.sum(y == 1)/len(y)*100:.1f}%)")

# First few features
print("\nFirst 5 feature names:")
print(data.feature_names[:5])

# Basic statistics of first feature
print(f"\nStatistics of '{data.feature_names[0]}':")
print(f"Mean: {X[:, 0].mean():.2f}")
print(f"Std: {X[:, 0].std():.2f}")
print(f"Min: {X[:, 0].min():.2f}")
print(f"Max: {X[:, 0].max():.2f}")


Class Distribution:
Malignant (0): 212 samples (37.3%)
Benign (1): 357 samples (62.7%)

First 5 feature names:
['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness']

Statistics of 'mean radius':
Mean: 14.13
Std: 3.52
Min: 6.98
Max: 28.11


**Important observations:**

1. **Imbalanced dataset**: We have more benign samples (62.7%) than malignant ones (37.3%)
2. **Numerical features**: All features are numerical, facilitating their use
3. **Wide scale variation**: Features have different scales (e.g., 'mean area' can be in thousands while 'mean smoothness' is between 0 and 1)

### 2.3 Manual Train-Test Split

In [3]:
# Set random seed for reproducibility
np.random.seed(42)

# Get total number of samples
n_samples = X.shape[0]

# Create array of indices and shuffle them
indices = np.arange(n_samples)
np.random.shuffle(indices)

# Calculate split point (80% train, 20% test)
split_point = int(0.8 * n_samples)

# Split indices
train_indices = indices[:split_point]
test_indices = indices[split_point:]

# Create training and test sets
X_train = X[train_indices]
X_test = X[test_indices]
y_train = y[train_indices]
y_test = y[test_indices]

print(f"Training samples: {len(X_train)} ({len(X_train)/n_samples*100:.1f}%)")
print(f"Test samples: {len(X_test)} ({len(X_test)/n_samples*100:.1f}%)")
print(f"\nTraining set class distribution:")
print(f"Malignant: {np.sum(y_train == 0)}, Benign: {np.sum(y_train == 1)}")
print(f"\nTest set class distribution:")
print(f"Malignant: {np.sum(y_test == 0)}, Benign: {np.sum(y_test == 1)}")

Training samples: 455 (80.0%)
Test samples: 114 (20.0%)

Training set class distribution:
Malignant: 165, Benign: 290

Test set class distribution:
Malignant: 47, Benign: 67


**Why this manual split?**

1. **Understanding**: You see exactly what `train_test_split` does internally
2. **Control**: You can modify the proportion or implement stratification
3. **Transparency**: No "magic" functions, everything is explicit

### 2.4 Feature Normalization

Normalization is **crucial** in Logistic Regression. Without it, features with large scales will dominate the gradient, making training unstable.

**Why normalize?**

Imagine two features:
- Feature 1: Values between 0 and 1
- Feature 2: Values between 0 and 10,000

During training, changes in Feature 2 will have 10,000 times more impact on the cost function, even if both features are equally important.

**Z-score normalization formula:**

$$x_{normalized} = \frac{x - \mu}{\sigma}$$

Where:
- $\mu$ is the mean
- $\sigma$ is the standard deviation

**Manual implementation:**

In [4]:
# Calculate mean and standard deviation from TRAINING set
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

# Normalize training set
X_train_normalized = (X_train - mean) / std

# Normalize test set using training statistics
X_test_normalized = (X_test - mean) / std

print("Before normalization:")
print(f"Training - Mean of first feature: {X_train[:, 0].mean():.2f}")
print(f"Training - Std of first feature: {X_train[:, 0].std():.2f}")

print("\nAfter normalization:")
print(f"Training - Mean of first feature: {X_train_normalized[:, 0].mean():.2f}")
print(f"Training - Std of first feature: {X_train_normalized[:, 0].std():.2f}")

Before normalization:
Training - Mean of first feature: 14.05
Training - Std of first feature: 3.48

After normalization:
Training - Mean of first feature: -0.00
Training - Std of first feature: 1.00


**CRITICAL:** Notice that we normalize the test set using the mean and standard deviation from the **training set**, not from the test set. This simulates what happens in production: we don't know the statistics of future data.

### 2.5 Data Verification

In [5]:
print("=" * 60)
print("DATA PREPARATION - FINAL VERIFICATION")
print("=" * 60)
print(f"X_train_normalized shape: {X_train_normalized.shape}")
print(f"X_test_normalized shape: {X_test_normalized.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"\nX_train_normalized - Min value: {X_train_normalized.min():.2f}")
print(f"X_train_normalized - Max value: {X_train_normalized.max():.2f}")
print(f"X_train_normalized - Mean: {X_train_normalized.mean():.2f}")
print(f"X_train_normalized - Std: {X_train_normalized.std():.2f}")
print("=" * 60)

DATA PREPARATION - FINAL VERIFICATION
X_train_normalized shape: (455, 30)
X_test_normalized shape: (114, 30)
y_train shape: (455,)
y_test shape: (114,)

X_train_normalized - Min value: -3.14
X_train_normalized - Max value: 11.36
X_train_normalized - Mean: 0.00
X_train_normalized - Std: 1.00


**Perfect!** Our data is now ready for training. In the next section, we will understand the mathematical foundations of the model.

---